In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import glob, os, yaml, itertools, subprocess, sys, shutil, re, vcf, pysam
from functools import reduce

import Bio.SeqUtils
import Bio.Data
from Bio import SeqIO, Entrez
from Bio.Seq import Seq

import scipy.stats as st
import statsmodels.stats.api as sm
import warnings, pickle
import xml.etree.ElementTree as ET
from dateutil import parser

import warnings
warnings.filterwarnings('ignore')

hybridASM_dir = "/n/data1/hms/dbmi/farhat/rollingDB/TRUST/PacBio_Illumina_hybridASM"
M_avium_ASM_dir = "/n/data1/hms/dbmi/farhat/rollingDB/TRUST/M_avium_ASMs"

df_redcap = pd.read_csv("~/MtbLongitudinalDiversity/TRUST_data_processing/raw_data/TRUST_DATA_2025-05-12_1129.cleaned.wide.csv")
sputum_collection_date_cols = list(df_redcap.columns[df_redcap.columns.str.contains('s_collectdate_sputum_specimen')])

In [28]:
df_full_SR.query("Original_ID in ['S0344-01', 'S0346-01']")

,pid,Original_ID,SampleID,SampleLib,Pacbio,Comment,quality,Cov Any Mean,Cov Unam Perc,Perc. Reads Mapped,...,duration_of_use_meth,duration_of_use_mandrax,lca,fib4,prevtb_outcome,F2,Coll2014,Freschi2020,Lineage,Sampling_Week
145,T0344,S0344-01,MFS-422,MFS-422_lib59429,NaN,NaN,4.0,37.95,0.01,0.2931,...,10.0,10.0,1.0,0.181046,NaN,NaN,NaN,NaN,NaN,1
152,T0346,S0346-01,MFS-425,MFS-425_lib59432,NaN,NaN,4.0,14.72,0.00,0.2240,...,NaN,NaN,1.0,1.048237,NaN,NaN,NaN,NaN,NaN,1


In [30]:
WGS_report_directory = "/n/data1/hms/dbmi/farhat/rollingDB/TRUST/WGS_metadata_reports"

# get all Excel files from this directory
trust_report_fNames = glob.glob(f"{WGS_report_directory}/*.xlsx")
print(f"{len(trust_report_fNames)} WGS metadata Excel files")

df_trust_WGS_metadata = []

# sort chronologically because below, we preferentially keep the later one
for fName in np.sort(trust_report_fNames)[::-1]:

    # read in single Excel file
    df = pd.read_excel(fName, sheet_name=None)

    # remove spaces from column name to make querying easier. Also there could be NaN rows if there are additional empty rows in the Excel sheet, so drop them
    df = df['summary'].rename(columns={'original ID': 'Original_ID'}).dropna(axis=0, how='all')

    # append to list for concatenation
    df_trust_WGS_metadata.append(df)

# the Excel files above are running totals, so the most recent file has data that is also in the older files. So drop duplicates, keeping the most recent (last) one
df_trust_WGS_metadata = pd.concat(df_trust_WGS_metadata).drop_duplicates('SampleID', keep='last')#.query("status!='failed'")


6 WGS metadata Excel files


In [32]:
df_trust_WGS_metadata.query("Original_ID in ['S0344-01', 'S0346-01']")

,SampleID,SampleLib,Original_ID,Pacbio,Comment,quality,Cov Any Mean,Cov Unam Perc,Perc. Reads Mapped,Excluded,...,Repeat_1_factor,repat_1_loading,Unnamed: 13,250x,repeat_pacbio,repeat_pcrfree,shipment,status,comment,"phylogenetic classification (Coll et al., 2014)"
417,MFS-422,MFS-422_lib59429,S0344-01,NaN,NaN,4.0,37.95,0.01,0.2931,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3rd,failed,-,4.4.1.2 Euro-American Pacific RD150
419,MFS-425,MFS-425_lib59432,S0346-01,NaN,NaN,4.0,14.72,0.0,0.224,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3rd,failed,-,4.6.2.1 Euro-American


In [43]:
df_trust_WGS_metadata['Original_ID'] = df_trust_WGS_metadata['Original_ID'].astype(str)
df_trust_WGS_metadata.dropna(subset='Original_ID').query("Original_ID.str.contains('S0346')")

,SampleID,SampleLib,Original_ID,Pacbio,Comment,quality,Cov Any Mean,Cov Unam Perc,Perc. Reads Mapped,Excluded,...,Repeat_1_factor,repat_1_loading,Unnamed: 13,250x,repeat_pacbio,repeat_pcrfree,shipment,status,comment,"phylogenetic classification (Coll et al., 2014)"
50,MFS-768,MFS-768_lib73027,S0346-02,NaN,NaN,1.0,232.97,0.99,0.9827,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51,MFS-769,MFS-769_lib73028,S0346-09,NaN,NaN,1.0,332.09,0.99,0.9847,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
282,MFS-426,MFS-426_lib59433,S0346-08,NaN,NaN,1.0,361.24,0.99,0.9885,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3rd,finished,-,2.2.1.1 Beijing Pacific RD150
419,MFS-425,MFS-425_lib59432,S0346-01,NaN,NaN,4.0,14.72,0.0,0.224,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3rd,failed,-,4.6.2.1 Euro-American


In [41]:
df_full_SR.query("SampleID in ['MFS-768', 'MFS-769', 'MFS-426', 'MFS-425']")

,pid,Original_ID,SampleID,SampleLib,Pacbio,Comment,quality,Cov Any Mean,Cov Unam Perc,Perc. Reads Mapped,...,duration_of_use_meth,duration_of_use_mandrax,lca,fib4,prevtb_outcome,F2,Coll2014,Freschi2020,Lineage,Sampling_Week
149,T0346,S0346-02,MFS-768,MFS-768_lib73027,NaN,NaN,1.0,232.97,0.99,0.9827,...,NaN,NaN,1.0,1.048237,NaN,0.011578,2.2.1.1,2.2.1.1.1,2.0,2
150,T0346,S0346-09,MFS-769,MFS-769_lib73028,NaN,NaN,1.0,332.09,0.99,0.9847,...,NaN,NaN,1.0,1.048237,NaN,0.012190,2.2.1,2.2.1.1.1,2.0,9
151,T0346,S0346-08,MFS-426,MFS-426_lib59433,NaN,NaN,1.0,361.24,0.99,0.9885,...,NaN,NaN,1.0,1.048237,NaN,0.011563,2.2.1.1,2.2.1.1.1,2.0,8
152,T0346,S0346-01,MFS-425,MFS-425_lib59432,NaN,NaN,4.0,14.72,0.00,0.2240,...,NaN,NaN,1.0,1.048237,NaN,NaN,NaN,NaN,NaN,1


In [27]:
df_full_SR = pd.read_csv("/n/data1/hms/dbmi/farhat/rollingDB/TRUST/clinical_data/combined_patient_WGS_data.csv")
df_LR_SR = pd.read_csv("~/SRA_Submissions/TRUST/LR_SR_mapped_data.csv")
df_LR_SR = df_LR_SR.merge(df_full_SR[['pid', 'SampleID']].rename(columns={'SampleID': 'Illumina_ID'}))

In [8]:
df_redcap.query("pid in ['T0346', 'T0344', 'T0360']")[['pid', 's_ntmsuspect_sputum_specimen_1', 's_ntmdate_sputum_specimen_1', 's_mtbassaydone_sputum_specimen_1', 's_mtbassay_sputum_specimen_1', 's_mtbconfirm_sputum_specimen_1', 's_ntmsuspect_sputum_specimen_2', 's_mtbassaydone_sputum_specimen_2', 'bl_hiv']]



,pid,s_ntmsuspect_sputum_specimen_1,s_ntmdate_sputum_specimen_1,s_mtbassaydone_sputum_specimen_1,s_mtbassay_sputum_specimen_1,s_mtbconfirm_sputum_specimen_1,s_ntmsuspect_sputum_specimen_2,s_mtbassaydone_sputum_specimen_2,bl_hiv
327,T0344,0.0,NaN,0.0,NaN,NaN,0.0,0.0,0.0
329,T0346,1.0,2022-07-12,1.0,CAPILIA,1.0,0.0,0.0,0.0
343,T0360,0.0,NaN,0.0,NaN,NaN,1.0,1.0,1.0


In [4]:
NTM_suspect_cols = list(df_redcap.columns[df_redcap.columns.str.contains('s_ntmsuspect_sputum_specimen')])
NTM_collection_date_cols = list(df_redcap.columns[df_redcap.columns.str.contains('s_ntmdate_sputum_specimen')])

In [156]:
df_redcap.columns[df_redcap.columns.str.contains('s_mtbconfirm')]

Index(['s_mtbconfirm_additional_visit_1', 's_mtbconfirm_additional_visit_2',
       's_mtbconfirm_sputum_specimen_1', 's_mtbconfirm_sputum_specimen_10',
       's_mtbconfirm_sputum_specimen_11', 's_mtbconfirm_sputum_specimen_12',
       's_mtbconfirm_sputum_specimen_13', 's_mtbconfirm_sputum_specimen_14',
       's_mtbconfirm_sputum_specimen_16', 's_mtbconfirm_sputum_specimen_17',
       's_mtbconfirm_sputum_specimen_18', 's_mtbconfirm_sputum_specimen_2',
       's_mtbconfirm_sputum_specimen_3', 's_mtbconfirm_sputum_specimen_4',
       's_mtbconfirm_sputum_specimen_5', 's_mtbconfirm_sputum_specimen_6',
       's_mtbconfirm_sputum_specimen_7', 's_mtbconfirm_sputum_specimen_8',
       's_mtbconfirm_sputum_specimen_9'],
      dtype='object')

# Three samples are heavily contaminated (< 25% MTBC reads)

In [133]:
kraken_reports = glob.glob(f"{hybridASM_dir}/*/PB/kraken/kraken_report_standard_DB.txt")
print(len(kraken_reports))

df_kraken_report = pd.DataFrame(columns=['Original_ID', 'Max_G1_Percent', 'Max_G1'])

for i, fName in enumerate(kraken_reports):
    df = pd.read_csv(fName, sep='\t', header=None)
    df.columns = ['Percent', 'CumulReads', 'NumReads', 'TaxGroup', 'TaxID', 'TaxName']
    
    sample = os.path.basename(os.path.dirname(os.path.dirname(os.path.dirname(fName))))
    
    df_kraken_report.loc[i, :] = [sample, float(df.query("TaxGroup=='G1'").Percent.values[0]), df.query("TaxGroup=='G1'").TaxName.values[0].strip()]

189


In [134]:
df_kraken_report.query("Original_ID=='S0346-08'")

,Original_ID,Max_G1_Percent,Max_G1
170,S0346-08,98.79,Mycobacterium tuberculosis complex


In [8]:
df_kraken_report.query("Max_G1 != 'Mycobacterium tuberculosis complex'")

,Original_ID,Max_G1_Percent,Max_G1
167,S0344-01,88.99,Mycobacterium avium complex (MAC)
169,S0346-01,97.95,Mycobacterium avium complex (MAC)
176,S0360-01,79.4,Mycobacterium avium complex (MAC)


In [162]:
df_kraken_report.query("Original_ID.str.contains('S0344')")

,Original_ID,Max_G1_Percent,Max_G1
167,S0344-01,88.99,Mycobacterium avium complex (MAC)


### Of these, S0360-01 did not have a clear NTM species. The other two are predominantly <i>M. avium</i>

<ul>
    <li>S0344-01: 89% <i>M. avium</i> complex, 89% <i>M. avium</i></li>
    <li>S0346-01: 98% <i>M. avium</i> complex, 98% <i>M. avium</i></li>
    <li>S0360-01: 79% <i>M. avium</i> complex, 30% <i>M. chimaera</i>, 29% <i>M. intracellulare</i>, etc.</li>
</ul>

In [90]:
sample = 'S0344-01'
pid = sample.split('-')[0].replace('S', 'T')

fName = f"{hybridASM_dir}/{sample}/PB/kraken/kraken_report_standard_DB.txt"
df = pd.read_csv(fName, sep='\t', header=None)
df.columns = ['Percent', 'CumulReads', 'NumReads', 'TaxGroup', 'TaxID', 'TaxName']
df['TaxName'] = df['TaxName'].str.strip()

df_full_SR.query("pid==@pid")

,pid,Original_ID,SampleID,SampleLib,Pacbio,Comment,quality,Cov Any Mean,Cov Unam Perc,Perc. Reads Mapped,...,duration_of_use_meth,duration_of_use_mandrax,lca,fib4,prevtb_outcome,F2,Coll2014,Freschi2020,Lineage,Sampling_Week
144,T0344,S0344-02,MFS-766,MFS-766_lib73025,NaN,NaN,1.0,294.31,0.99,0.9822,...,10.0,10.0,1.0,0.181046,NaN,0.011588,2.2.1.1,2.2.1.1.1,2.0,2
145,T0344,S0344-01,MFS-422,MFS-422_lib59429,NaN,NaN,4.0,37.95,0.01,0.2931,...,10.0,10.0,1.0,0.181046,NaN,NaN,NaN,NaN,NaN,1


In [91]:
MFS_ID = 'MFS-766'
fName = f"/n/data1/hms/dbmi/farhat/rollingDB/TRUST/Illumina_culture_WGS_processed/{MFS_ID}/{MFS_ID}/kraken/kraken_report_standard_DB"

# only the highly contaminated ones have a standard kraken DB report in the above directory
# if it wasn't highly contaminated, then it's in the following directory
if not os.path.isfile(fName):
    fName = f"/n/data1/hms/dbmi/farhat/Sanjana/TRUST_lowAF/{MFS_ID}/{MFS_ID}/kraken/kraken_report_standard_DB.txt"

df = pd.read_csv(fName, sep='\t', header=None)
df.columns = ['Percent', 'CumulReads', 'NumReads', 'TaxGroup', 'TaxID', 'TaxName']
df['TaxName'] = df['TaxName'].str.strip()

In [95]:
df.query("TaxGroup=='S'").query("TaxName.str.contains('Mycobacterium avium')")

,Percent,CumulReads,NumReads,TaxGroup,TaxID,TaxName
172,0.01,520,286,S,1764,Mycobacterium avium


In [94]:
df.query("TaxGroup=='S'")

,Percent,CumulReads,NumReads,TaxGroup,TaxID,TaxName
11,10.76,505505,427450,S,1773,Mycobacterium tuberculosis
155,0.70,32985,11187,S,78331,Mycobacterium canettii
162,0.02,1025,1025,S,53376,Mycobacterium heidelbergense
163,0.02,912,912,S,220927,Mycobacterium saskatchewanense
164,0.01,653,653,S,292462,Mycobacterium florentinum
...,...,...,...,...,...,...
541,0.00,1,1,S,114,Gemmata obscuriglobus
549,0.00,1,0,S,328515,Nonlabens dokdonensis
580,0.51,24062,24062,S,9606,Homo sapiens
588,0.00,1,1,S,35746,Haloferax gibbonsii


### Extracted <i>M. avium</i> reads (taxid 1764 and parents and children) with kraken2 and assembled with flye

### for S0344-01 and S0346-01 only

In [44]:
MAv_assembly_reports = glob.glob(f"{M_avium_ASM_dir}/*/PB/Flye_Assembly/assembly_info.txt")
len(MAv_assembly_reports)

2

## One circular contig of 5.6 Mbp each

In [45]:
df_MAv_assemblies = {}

for fName in MAv_assembly_reports:
    
    sample = re.findall(r"S[a-zA-Z0-9]\d{3}-\d{2}", fName)
    assert len(sample) == 1
    sample = sample[0]

    df_MAv_report = pd.read_csv(fName, sep='\t')
    df_MAv_report['Original_ID'] = sample
    df_MAv_assemblies[sample] = df_MAv_report

In [46]:
df_MAv_assemblies['S0344-01']

,#seq_name,length,cov.,circ.,repeat,mult.,alt_group,graph_path,Original_ID
0,contig_1,5575294,103,Y,N,1,*,1,S0344-01


In [47]:
df_MAv_assemblies['S0346-01']

,#seq_name,length,cov.,circ.,repeat,mult.,alt_group,graph_path,Original_ID
0,contig_1,5573596,94,Y,N,1,*,1,S0346-01


# Check Number of Pilon Changes after polishing with short reads

In [18]:
df_pilon_changes

,Contig1_POS,Contig2_POS,OG,CHANGE
0,S0344-01_contig_1:1505017,S0344-01_contig_1_pilon:1505017,T,.
1,S0344-01_contig_1:5231822,S0344-01_contig_1_pilon:5231821,C,.


In [22]:
sample = 'S0344-01'
pilon_changes_file = f"{M_avium_ASM_dir}/{sample}/FlyeAssembly_I3_PilonPolishing/pilon_IllPE_Polishing_I3_Asm_ChangeSNPsINDELsOnly/{sample}.Flye.I3Asm.PilonPolished.changes"

try:
    df_pilon_changes = pd.read_csv(pilon_changes_file, sep=' ', header=None)
    df_pilon_changes.columns = ['Contig1_POS', 'Contig2_POS', 'OG', 'CHANGE']
    print(f'{len(df_pilon_changes.query("OG.str.len() == CHANGE.str.len()"))} SNV changes, {len(df_pilon_changes.query("OG.str.len() != CHANGE.str.len()"))} indel changes')
except:
    assert os.path.isfile(pilon_changes_file)
    print(f"No pilon changes")

2 SNV changes, 0 indel changes


In [23]:
sample = 'S0346-01'
pilon_changes_file = f"{M_avium_ASM_dir}/{sample}/FlyeAssembly_I3_PilonPolishing/pilon_IllPE_Polishing_I3_Asm_ChangeSNPsINDELsOnly/{sample}.Flye.I3Asm.PilonPolished.changes"

try:
    df_pilon_changes = pd.read_csv(pilon_changes_file, sep=' ', header=None)
    df_pilon_changes.columns = ['Contig1_POS', 'Contig2_POS', 'OG', 'CHANGE']
    print(f'{len(df_pilon_changes.query("OG.str.len() == CHANGE.str.len()"))} SNV changes, {len(df_pilon_changes.query("OG.str.len() != CHANGE.str.len()"))} indel changes')
except:
    assert os.path.isfile(pilon_changes_file)
    print(f"No pilon changes")

No pilon changes


In [14]:
S0346_genome_fName = f"{M_avium_ASM_dir}/S0346-01/FlyeAssembly_I3_PilonPolishing/pilon_IllPE_Polishing_I3_Asm_ChangeSNPsINDELsOnly/S0346-01.Flye.I3Asm.PilonPolished.fasta"
S0346_genome = [(seq.id, seq.seq) for seq in SeqIO.parse(S0346_genome_fName, "fasta")]
S0346_genome = S0346_genome[0][1]

S0344_genome_fName = f"{M_avium_ASM_dir}/S0344-01/FlyeAssembly_I3_PilonPolishing/pilon_IllPE_Polishing_I3_Asm_ChangeSNPsINDELsOnly/S0344-01.Flye.I3Asm.PilonPolished.fasta"
S0344_genome = [(seq.id, seq.seq) for seq in SeqIO.parse(S0344_genome_fName, "fasta")]
S0344_genome = S0344_genome[0][1]

len(S0346_genome), len(S0344_genome), abs(len(S0346_genome) - len(S0344_genome))

(5573596, 5575292, 1696)

In [27]:
# the top hit for this sequence (BLAST) is an IS1380 transposase (% identity of 97.52% to WP_462939999.1)
# so these two assemblies seem to differ by a single additional IS1380 sequences
str(S0344_genome[2234622:2236317])

'GGCTAGCCTCGATTTTGCATTGAGTTGGTGCGGAGTCGGTTCTGCCTGTGGATTGTGTTGCCGGGTAGGTGTTTTCGTGTGATGGCGTGGTGGGGCGTCCCGTGTCGCCGGGTGGGCGGGCTTTCCCAGGGCCTGGTCTTTCTGATCGGTGGGTCAGGTGAGCCGGGTTCTATCCGAAGGCGGTGCGTAGGTGTTGCCAGGCGGTCGCGATCGCTGATGCCCAGCGCCAGGTGGCATCGATACGTAGTCGTTGTTGGCGGGCGCCGCGAGTGATACGGGCAGCGACGTGCAGGACCCGGTAGCGGAAGGTGGCGATCTCGGCGCGAGCCAGGCCGGGATGTGCGCGGAAGCCGATCAGTCGGGCCCAGGTGACCAGATCGGCGGCGGCCAGCACGATTTCCAGCCAGGCGGCGTTGGCCCAGAACGAGTGGCACGGCAGGTTACGCAGGCCGGTGGCTTTGAGTTCGCGGATGCGGTCCTCGACGCGGGCGTGCTGGCGGTGCCGCAATTCCAGACCGGCGACCTGACCAGCAATGACACCGGGTTCGGTGTCGGTGATGAACGCGGTGACCCGCATCCCGTCGGCGTCGGTGAACCGCAACTGCGCACCGGGATGCGGGCGTTCTTTGCGCAGGATCAGCCGGGTGCCGGCCGGCCAGCTACTGAGGTTGACCAGGTCGGTGGCCTCGGCGACCCAGGCCCCCTCGCGGATCCCGCCGTCGGTGTCGATCGCTGGATACCAGCCATCGGCGAGGTTGAGGGTGTCCACCGCGTCCTGGACGCGCACATCGACGGGGTAGCCGAAGGAGAACCCGACCCCGGCGGCGCGGCAGGCGTCGGCGAACTTGTGGGTGGCTCCGGCAGTGTCGCAGCGCACCAACACCCCAGACGCGTCCGGATCCGCCCGAGCACCTGGGTGGGGCCGCCATCGCGGCGGCAGCGACGCCAGCGCTTGTTTGAGGACGATGATGTGATCGGAGGCGGTGTTGGAGCCGGCGT

## 354/356 BUSCO groups are complete and single-copy for both

## 99.9976% similarity to each other (ANI is computed only among orthologous regions)

In [25]:
MAv_genome_dir = "/n/data1/hms/dbmi/farhat/Sanjana/MAv_genomes"

MAv_genomes = glob.glob(f"{MAv_genome_dir}/*")
print(len(MAv_genomes))

pd.Series(MAv_genomes).to_csv("~/MtbLongitudinalDiversity/hybrid_assemblies/MAv/MAv_NCBI_genomes.txt", sep='\t', header=None, index=False)

86


In [26]:
fastANI_out = pd.read_csv(f"{M_avium_ASM_dir}/S0346-01/FlyeAssembly_I3_PilonPolishing/pilon_IllPE_Polishing_I3_Asm_ChangeSNPsINDELsOnly/fastANI_NCBI_genomes.out", sep='\t', header=None)
fastANI_out.columns = ['Query', 'Reference', 'ANI', 'Matches', 'TotalFragments']
fastANI_out['FASTA'] = [os.path.basename(fName) for fName in fastANI_out.Reference.values]

genome_lengths = pd.read_csv("~/MtbLongitudinalDiversity/hybrid_assemblies/MAv/public_genome_lengths.txt", sep='\t', header=None)
genome_lengths.columns = ['Reference', 'Length']

fastANI_out = fastANI_out.merge(genome_lengths, how='outer')

len(fastANI_out)

172

In [29]:
f"{M_avium_ASM_dir}/S0346-01/FlyeAssembly_I3_PilonPolishing/pilon_IllPE_Polishing_I3_Asm_ChangeSNPsINDELsOnly"

'/n/data1/hms/dbmi/farhat/rollingDB/TRUST/M_avium_ASMs/S0346-01/FlyeAssembly_I3_PilonPolishing/pilon_IllPE_Polishing_I3_Asm_ChangeSNPsINDELsOnly'

In [27]:
fastANI_out.ANI.min(), fastANI_out.ANI.max()

(98.2584, 99.6495)

In [28]:
fastANI_out.sort_values('ANI', ascending=False)

,Query,Reference,ANI,Matches,TotalFragments,FASTA,Length
0,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/M_avi...,/n/data1/hms/dbmi/farhat/Sanjana/MAv_genomes/G...,99.6495,1634,1857,GCA_057005245.1_ASM5700524v1_genomic.fna,5055633
1,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/M_avi...,/n/data1/hms/dbmi/farhat/Sanjana/MAv_genomes/G...,99.6495,1634,1857,GCA_057005245.1_ASM5700524v1_genomic.fna,5055633
2,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/M_avi...,/n/data1/hms/dbmi/farhat/Sanjana/MAv_genomes/G...,99.6494,1634,1857,GCA_056355635.1_ASM5635563v1_genomic.fna,5055634
3,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/M_avi...,/n/data1/hms/dbmi/farhat/Sanjana/MAv_genomes/G...,99.6494,1634,1857,GCA_056355635.1_ASM5635563v1_genomic.fna,5055634
4,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/M_avi...,/n/data1/hms/dbmi/farhat/Sanjana/MAv_genomes/G...,99.6400,1635,1857,GCA_056337065.1_ASM5633706v1_genomic.fna,5057098
...,...,...,...,...,...,...,...
167,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/M_avi...,/n/data1/hms/dbmi/farhat/Sanjana/MAv_genomes/G...,98.3160,1638,1857,GCA_002304065.2_ASM230406v2_genomic.fna,5388949
168,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/M_avi...,/n/data1/hms/dbmi/farhat/Sanjana/MAv_genomes/G...,98.2820,1621,1857,GCA_055386175.1_ASM5538617v1_genomic.fna,5607771
169,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/M_avi...,/n/data1/hms/dbmi/farhat/Sanjana/MAv_genomes/G...,98.2820,1621,1857,GCA_055386175.1_ASM5538617v1_genomic.fna,5607771
170,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/M_avi...,/n/data1/hms/dbmi/farhat/Sanjana/MAv_genomes/G...,98.2584,1621,1857,GCA_038446055.1_ASM3844605v1_genomic.fna,5527856


In [131]:
# this is the M. avium 104 reference sequence
fastANI_out.query("FASTA.str.contains('GCA_000014985.1')")

,Query,Reference,ANI,Matches,TotalFragments,FASTA,Length
3,/n/data1/hms/dbmi/farhat/rollingDB/TRUST/M_avi...,/n/data1/hms/dbmi/farhat/Sanjana/MAv_genomes/G...,99.6291,1752,1857,GCA_000014985.1_ASM1498v1_genomic.fna,5475491


In [22]:
sam_fName = "/home/sak0914/MtbLongitudinalDiversity/hybrid_assemblies/MAv/two_assembly_comparison/S0346-01_S0344-01.sam"
samfile = pysam.AlignmentFile(sam_fName, "r")

for read in samfile.fetch():
    print(
        read.query_name,
        read.reference_name,
        read.reference_start,
        len(read.query_sequence),
        read.mapping_quality
    )

S0346-01_contig_1_pilon S0344-01_contig_1_pilon 0 5573596 60


In [23]:
sam_fName = "/home/sak0914/MtbLongitudinalDiversity/hybrid_assemblies/MAv/two_assembly_comparison/S0344-01_S0346-01.sam"
samfile = pysam.AlignmentFile(sam_fName, "r")

for read in samfile.fetch():
    print(
        read.query_name,
        read.reference_name,
        read.reference_start,
        len(read.query_sequence),
        read.mapping_quality
    )

S0344-01_contig_1_pilon S0346-01_contig_1_pilon 0 5575292 60


# Genome Annotations with bakta

```bash
cd /n/data1/hms/dbmi/farhat/rollingDB/TRUST/M_avium_ASMs/S0344-01/FlyeAssembly_I3_PilonPolishing/pilon_IllPE_Polishing_I3_Asm_ChangeSNPsINDELsOnly
bakta --db /n/scratch/users/s/sak0914/databases/Bakta/db S0344-01.Flye.I3Asm.PilonPolished.fasta --output bakta --genus Mycobacterium --species avium --prefix annot

cd /n/data1/hms/dbmi/farhat/rollingDB/TRUST/M_avium_ASMs/S0346-01/FlyeAssembly_I3_PilonPolishing/pilon_IllPE_Polishing_I3_Asm_ChangeSNPsINDELsOnly
bakta --db /n/scratch/users/s/sak0914/databases/Bakta/db S0346-01.Flye.I3Asm.PilonPolished.fasta --output bakta --genus Mycobacterium --species avium --prefix annot
```

In [13]:
gff_columns = [
    "seqid",
    "source",
    "feature_type",
    "start",
    "end",
    "score",
    "strand",
    "phase",
    "attributes"
]

sample = 'S0344-01'
gff_file = f"{M_avium_ASM_dir}/{sample}/FlyeAssembly_I3_PilonPolishing/pilon_IllPE_Polishing_I3_Asm_ChangeSNPsINDELsOnly/bakta/annot.gff3"

S0344_01_gff = pd.read_csv(
    gff_file,
    sep="\t",
    comment="#",
    header=None,
    names=gff_columns
)

sample = 'S0346-01'
gff_file = f"{M_avium_ASM_dir}/{sample}/FlyeAssembly_I3_PilonPolishing/pilon_IllPE_Polishing_I3_Asm_ChangeSNPsINDELsOnly/bakta/annot.gff3"

S0346_01_gff = pd.read_csv(
    gff_file,
    sep="\t",
    comment="#",
    header=None,
    names=gff_columns
)

S0344_01_gff['length'] = S0344_01_gff['end'] - S0344_01_gff['start'] + 1
S0346_01_gff['length'] = S0346_01_gff['end'] - S0346_01_gff['start'] + 1

len(S0344_01_gff.query("feature_type=='CDS'").query("~attributes.str.contains('RNA')").query("~attributes.str.contains('pseudo')")), len(S0346_01_gff.query("feature_type=='CDS'").query("~attributes.str.contains('RNA')").query("~attributes.str.contains('pseudo')"))

(5142, 5141)

In [78]:
S0344_01_gff.query("start >= 2234622 & start <= 2236317")

,seqid,source,feature_type,start,end,score,strand,phase,attributes
2209,contig_1,Pyrodigal,CDS,2234792.0,2236198.0,.,-,0,ID=GFLMJF_02180;Name=IS1380 family ISMav7 tran...


In [5]:
S0344_01_gff.query("start >= 2234622 & start <= 2236317")

,seqid,source,feature_type,start,end,score,strand,phase,attributes,length
2209,contig_1,Pyrodigal,CDS,2234792.0,2236198.0,.,-,0,ID=GFLMJF_02180;Name=IS1380 family ISMav7 tran...,1407.0


In [6]:
S0344_01_gff.iloc[2208:2212]

,seqid,source,feature_type,start,end,score,strand,phase,attributes,length
2208,contig_1,Pyrodigal,CDS,2233790.0,2234575.0,.,+,0,ID=GFLMJF_02179;Name=Pyridoxal phosphate homeo...,786.0
2209,contig_1,Pyrodigal,CDS,2234792.0,2236198.0,.,-,0,ID=GFLMJF_02180;Name=IS1380 family ISMav7 tran...,1407.0
2210,contig_1,Pyrodigal,CDS,2236337.0,2236981.0,.,+,0,ID=GFLMJF_02181;Name=Cell division protein Sep...,645.0
2211,contig_1,Pyrodigal,CDS,2237094.0,2237441.0,.,+,0,ID=GFLMJF_02182;Name=YggT;locus_tag=GFLMJF_021...,348.0


In [79]:
S0344_01_gff.iloc[2208:2212]

,seqid,source,feature_type,start,end,score,strand,phase,attributes
2208,contig_1,Pyrodigal,CDS,2233790.0,2234575.0,.,+,0,ID=GFLMJF_02179;Name=Pyridoxal phosphate homeo...
2209,contig_1,Pyrodigal,CDS,2234792.0,2236198.0,.,-,0,ID=GFLMJF_02180;Name=IS1380 family ISMav7 tran...
2210,contig_1,Pyrodigal,CDS,2236337.0,2236981.0,.,+,0,ID=GFLMJF_02181;Name=Cell division protein Sep...
2211,contig_1,Pyrodigal,CDS,2237094.0,2237441.0,.,+,0,ID=GFLMJF_02182;Name=YggT;locus_tag=GFLMJF_021...


In [7]:
len(S0344_01_gff.dropna(subset='attributes').query("attributes.str.contains('IS')")), len(S0346_01_gff.dropna(subset='attributes').query("attributes.str.contains('IS')"))

(158, 157)

In [8]:
len(S0344_01_gff.dropna(subset='attributes').query("attributes.str.contains('IS1380')")), len(S0346_01_gff.dropna(subset='attributes').query("attributes.str.contains('IS1380')"))

(2, 1)

In [9]:
S0344_01_gff.dropna(subset='attributes').query("attributes.str.contains('IS1380')")

,seqid,source,feature_type,start,end,score,strand,phase,attributes,length
2209,contig_1,Pyrodigal,CDS,2234792.0,2236198.0,.,-,0,ID=GFLMJF_02180;Name=IS1380 family ISMav7 tran...,1407.0
3239,contig_1,Pyrodigal,CDS,3399907.0,3401313.0,.,+,0,ID=GFLMJF_03203;Name=IS1380 family ISMav7 tran...,1407.0


In [10]:
S0344_01_gff.iloc[2208:2210]

,seqid,source,feature_type,start,end,score,strand,phase,attributes,length
2208,contig_1,Pyrodigal,CDS,2233790.0,2234575.0,.,+,0,ID=GFLMJF_02179;Name=Pyridoxal phosphate homeo...,786.0
2209,contig_1,Pyrodigal,CDS,2234792.0,2236198.0,.,-,0,ID=GFLMJF_02180;Name=IS1380 family ISMav7 tran...,1407.0


In [15]:
S0344_genome[2234575-1:2234792][-10:]

Seq('AGCCGGGTTC')

In [16]:
S0344_genome[2236198:]

Seq('GTGCGAAGTGCCTTCCAGCTGGAACGATCTGAACCTTAGATAAGTCCGATTATC...TCG')

In [17]:
S0346_01_gff.dropna(subset='attributes').query("attributes.str.contains('IS1380')")

,seqid,source,feature_type,start,end,score,strand,phase,attributes,length
3238,contig_1,Pyrodigal,CDS,3398211.0,3399617.0,.,+,0,ID=DBDHAP_03202;Name=IS1380 family ISMav7 tran...,1407.0


In [18]:
IS1380_1 = S0344_genome[2234792-1:2236198].reverse_complement()
IS1380_2 = S0344_genome[3399907-1:3401313]
IS1380_1 == IS1380_2

True